# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and the skill in `skills/mbse-schemas/` teach AI agents what this package is for and how to
use it. Stale guidance misleads more than missing guidance, so these cases check that the packaged copy of the skill
is current, that its complete programs run, and that every link in the guides resolves.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

import mbse.Schemas.Framework as framework

ROOT = Path.cwd().parent.parent
SKILL = ROOT / "skills" / "mbse-schemas"
PACKAGED = Path(framework.__file__).parent / "skill"
REPO = "https://github.com/pitaman71/mbse-schemas/blob/main/"


def files(folder):
    """The files under `folder`, by relative path."""
    return {p.relative_to(folder).as_posix(): p.read_bytes() for p in sorted(folder.rglob("*")) if p.is_file()}


GUIDES = [ROOT / "AGENTS.md", ROOT / "llms.txt", *(SKILL / name for name in files(SKILL) if name.endswith(".md"))]


def blocks(language):
    """The complete programs in the skill: fenced blocks whose info string is exactly `language`."""
    return [code for guide in GUIDES[2:] for code in re.findall(rf"^```{language}\n(.*?)^```$", guide.read_text(), re.M | re.S)]
LINKS = 37

## SKL-01 · The packaged copy of the skill is current

In [ ]:
assert "SKILL.md" in files(SKILL)
assert files(PACKAGED) == files(SKILL), "the packaged skill is stale: run skills/sync.sh"

## SKL-02 · Every complete program in the skill runs

In [ ]:
programs = blocks("python")
assert len(programs) == 1, len(programs)
failures = {}
for i, code in enumerate(programs):
    result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        failures[i] = result.stderr[-2000:]
assert failures == {}, failures

## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
def slug(heading):
    """GitHub's anchor for a heading."""
    return re.sub(r"[^\w\- ]", "", heading.strip().lower()).replace(" ", "-")


problems, checked = [], 0
for guide in GUIDES:
    for target in re.findall(r"\]\(([^)\s]+)\)", guide.read_text()):
        if target.startswith(REPO):
            path, base = target[len(REPO):], ROOT
        elif "://" in target:
            continue
        else:
            path, base = target, guide.parent
        name, _, anchor = path.partition("#")
        resolved = base / name
        checked += 1
        if not resolved.exists():
            problems.append(f"{guide.name}: {target}")
        elif anchor and anchor not in {slug(h) for h in re.findall(r"^#+ (.*)$", resolved.read_text(), re.M)}:
            problems.append(f"{guide.name}: {target}")
assert problems == [], problems
assert checked == LINKS, checked